In [0]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %run ../00_setup/00_adls_gen2_oauth_connection

# COMMAND ----------
# ==============================
# CONFIG
# ==============================
from datetime import datetime
import os
import shutil
import zipfile
from pyspark.sql.functions import current_timestamp, input_file_name, lit

storage_account = "stspmobilitydev001"
container = "bronze"

ingestion_date = datetime.now().strftime("%Y-%m-%d")
run_id = datetime.now().strftime("%Y%m%d%H%M%S")

zip_source_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs/raw/cittamobi_gtfs.zip"
zip_local_path = f"/tmp/cittamobi_gtfs_{run_id}.zip"
extract_path = f"/tmp/gtfs_extract_{run_id}"

print("=== CONFIG ===")
print(f"zip_source_path: {zip_source_path}")

# COMMAND ----------
# ==============================
# VALIDAÇÃO DO ARQUIVO
# ==============================
print("Validando existência do ZIP...")

try:
    files = dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/gtfs/raw/")
    print("Arquivos encontrados:")
    for f in files:
        print(f.name)
except Exception as e:
    raise Exception(f"Erro ao acessar ADLS: {e}")

# COMMAND ----------
# ==============================
# COPIAR ZIP PARA LOCAL
# ==============================
print("Copiando ZIP para /tmp...")

dbutils.fs.cp(zip_source_path, f"file:{zip_local_path}", True)

print("ZIP copiado com sucesso")

# COMMAND ----------
# ==============================
# EXTRAÇÃO
# ==============================
print("Extraindo ZIP...")

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_local_path, "r") as z:
    z.extractall(extract_path)

print("Arquivos extraídos:")
print(os.listdir(extract_path))

# COMMAND ----------
# ==============================
# FUNÇÃO DE LEITURA
# ==============================
def read_gtfs(file_name):
    path = os.path.join(extract_path, file_name)
    
    if not os.path.exists(path):
        raise Exception(f"Arquivo não encontrado no ZIP: {file_name}")
    
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"file://{path}")
        .withColumn("ingestion_date", lit(ingestion_date))
        .withColumn("ingested_at", current_timestamp())
    )
    
    print(f"{file_name}: {df.count()} registros")
    return df

# COMMAND ----------
# ==============================
# LEITURA DOS ARQUIVOS GTFS
# ==============================
routes_df = read_gtfs("routes.txt")
trips_df = read_gtfs("trips.txt")
stops_df = read_gtfs("stops.txt")
stop_times_df = read_gtfs("stop_times.txt")
calendar_df = read_gtfs("calendar.txt")
shapes_df = read_gtfs("shapes.txt")

# COMMAND ----------
# ==============================
# CAMINHOS DELTA (BRONZE)
# ==============================
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

delta_paths = {
    "routes": f"{base_path}/gtfs_routes",
    "trips": f"{base_path}/gtfs_trips",
    "stops": f"{base_path}/gtfs_stops",
    "stop_times": f"{base_path}/gtfs_stop_times",
    "calendar": f"{base_path}/gtfs_calendar",
    "shapes": f"{base_path}/gtfs_shapes",
}

# COMMAND ----------
# ==============================
# GRAVAÇÃO DELTA
# ==============================
print("Gravando em Delta...")

routes_df.write.format("delta").mode("overwrite").save(delta_paths["routes"])
trips_df.write.format("delta").mode("overwrite").save(delta_paths["trips"])
stops_df.write.format("delta").mode("overwrite").save(delta_paths["stops"])
stop_times_df.write.format("delta").mode("overwrite").save(delta_paths["stop_times"])
calendar_df.write.format("delta").mode("overwrite").save(delta_paths["calendar"])
shapes_df.write.format("delta").mode("overwrite").save(delta_paths["shapes"])

print("Dados gravados com sucesso")

# COMMAND ----------
# ==============================
# VALIDAÇÃO FINAL
# ==============================
print("Validando escrita...")

for name, path in delta_paths.items():
    try:
        dbutils.fs.ls(path)
        print(f"{name}: OK")
    except:
        print(f"{name}: ERRO")

print("=== INGESTÃO FINALIZADA ===")

# COMMAND ----------
# ==============================
# LIMPEZA
# ==============================
print("Limpando arquivos temporários...")

try:
    os.remove(zip_local_path)
    shutil.rmtree(extract_path)
except:
    pass

print("Limpeza concluída")